In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from math import atan2, pi

def fast_cosine_similarity(a, b):
    a = np.asarray(a).ravel() 
    b = np.asarray(b).ravel()
    dot = np.dot(a, b)
    norm = np.linalg.norm(a) * np.linalg.norm(b)
    return dot / norm if norm > 0 else 0.0

    
def calculate_loop_features(coords):
    if len(coords) < 4:
        return {
            'loop_score': np.nan,
            'angle_variation': np.nan,
            'curvature_count': np.nan,
            'polygon_area': np.nan,
            'turning_index': np.nan
        }

    start, end = coords[0], coords[-1]
    path_length = np.sum(np.linalg.norm(np.diff(coords, axis=0), axis=1))
    loop_score = np.linalg.norm(end - start) / path_length if path_length > 0 else np.nan

    angles = []
    for i in range(1, len(coords) - 1):
        v1, v2 = coords[i] - coords[i - 1], coords[i + 1] - coords[i]
        angle = atan2(v2[1], v2[0]) - atan2(v1[1], v1[0])
        angle = (angle + pi) % (2 * pi) - pi
        angles.append(angle)

    angle_variation = np.std(angles)
    turning_index = abs(np.sum(angles)) / (2 * pi)
    curvature_count = np.sum(np.abs(angles) > (pi / 2))

    x, y = coords[:, 0], coords[:, 1]
    polygon_area = 0.5 * np.abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))

    return {
        'loop_score': loop_score,
        'angle_variation': angle_variation,
        'curvature_count': curvature_count,
        'polygon_area': polygon_area,
        'turning_index': turning_index
    }
def calculate_turns(window_traj):
    turn_count = 0
    for n in range(1, len(window_traj) - 1):
        prev_x, prev_y = window_traj[n - 1]
        curr_x, curr_y = window_traj[n]
        next_x, next_y = window_traj[n + 1]
        v1 = np.array([curr_x - prev_x, curr_y - prev_y])
        v2 = np.array([next_x - curr_x, next_y - curr_y])
        angle = np.arctan2(v2[1], v2[0]) - np.arctan2(v1[1], v1[0])
        if np.abs(np.degrees(angle)) > 60:
            turn_count += 1
    return turn_count

def calculate_neighbor_similarity(cell_subdf, df_all, radius=100, k=5):
    cs = cell_subdf.copy()
    cs['frame'] = cs['frame'].astype(int)
    cs['x_position'] = cs['x_position'].astype(float)
    cs['y_position'] = cs['y_position'].astype(float)
    cs['label_clean'] = cs['label'].astype(str).str.strip().str.rstrip('.0')

    similarities = []

    for i in range(1, len(cs)):
        row_t   = cs.iloc[i]
        row_t_1 = cs.iloc[i-1]

        frame = int(row_t['frame'])
        x, y  = float(row_t['x_position']),  float(row_t['y_position'])
        px,py = float(row_t_1['x_position']), float(row_t_1['y_position'])

        curr_vector = np.array([x - px, y - py], dtype=float)
        if np.linalg.norm(curr_vector) == 0:
            continue

        df_frame = df_all[df_all['frame'] == frame]
        df_frame = df_frame[df_frame['label_clean'] != row_t['label_clean']]

        if df_frame.empty:
            continue

        dx = df_frame['x_position'].to_numpy() - x
        dy = df_frame['y_position'].to_numpy() - y
        dist = np.sqrt(dx**2 + dy**2)
        df_frame = df_frame.assign(distance=dist)
        nearby = df_frame[df_frame['distance'] <= radius].nsmallest(k, 'distance')
        if nearby.empty:
            continue

        sim_list = []
        for _, nei in nearby.iterrows():
            nei_label = nei['label_clean']

            prev_nei = df_all[(df_all['label_clean'] == nei_label) & (df_all['frame'] == frame-1)]
            if prev_nei.empty:
                continue

            nx_t,  ny_t  = float(nei['x_position']),            float(nei['y_position'])
            nx_t1, ny_t1 = float(prev_nei.iloc[0]['x_position']), float(prev_nei.iloc[0]['y_position'])
            nei_vector = np.array([nx_t - nx_t1, ny_t - ny_t1], dtype=float)

            if np.linalg.norm(nei_vector) == 0:
                continue

            sim = fast_cosine_similarity(curr_vector, nei_vector)
            sim_list.append(sim)

        if sim_list:
            similarities.append(np.mean(sim_list))

    return np.mean(similarities) if similarities else 0.0

    for i in range(1, len(cs)):
        row_t   = cs.iloc[i]
        row_t_1 = cs.iloc[i-1]

        frame = int(row_t['frame'])
        x, y  = float(row_t['x_position']),  float(row_t['y_position'])
        px,py = float(row_t_1['x_position']), float(row_t_1['y_position'])

        curr_vector = np.array([x - px, y - py], dtype=float)
        if np.linalg.norm(curr_vector) == 0:
            continue  

        df_frame = df_all[df_all['frame'] == frame]
        # except reference
        df_frame = df_frame[df_frame['label_clean'] != row_t['label_clean']]

        if df_frame.empty:
            continue

        # filter for searching
        dx = df_frame['x_position'].to_numpy() - x
        dy = df_frame['y_position'].to_numpy() - y
        dist = np.sqrt(dx**2 + dy**2)
        df_frame = df_frame.assign(distance=dist)
        nearby = df_frame[df_frame['distance'] <= radius].nsmallest(k, 'distance')
        if nearby.empty:
            continue

        sim_list = []
        for _, nei in nearby.iterrows():
            nei_label = nei['label_clean']

            # Neigboring vectors
            prev_nei = df_all[(df_all['label_clean'] == nei_label) & (df_all['frame'] == frame-1)]
            if prev_nei.empty:
                continue  # Skip data without previous positions

            nx_t,  ny_t  = float(nei['x_position']),            float(nei['y_position'])
            nx_t1, ny_t1 = float(prev_nei.iloc[0]['x_position']), float(prev_nei.iloc[0]['y_position'])
            nei_vector = np.array([nx_t - nx_t1, ny_t - ny_t1], dtype=float)

            if np.linalg.norm(nei_vector) == 0:
                continue

            sim = fast_cosine_similarity(curr_vector, nei_vector)  
            sim_list.append(sim)

        if sim_list:
            similarities.append(np.mean(sim_list))

    return np.mean(similarities) if similarities else 0.0

def extract_migration_properties(cell_tarray, t_window=5, sampling_interval=1, arrest_threshold=0.5, df_in=None, min_valid_points=1):
    cell_calcs = []
    turn_count = calculate_turns(cell_tarray[:, 1:3])
    if cell_tarray.shape[0] >= min_valid_points:
        init_f = int(np.min(cell_tarray[:, 0]))
        final_f = int(np.max(cell_tarray[:, 0]))
        for t in range(init_f + 1, final_f):
            prev_frame_arr = cell_tarray[cell_tarray[:, 0] == t - 1]
            this_frame_arr = cell_tarray[cell_tarray[:, 0] == t]
            t_window_arr = cell_tarray   

            if len(this_frame_arr) == 0 or len(prev_frame_arr) == 0:
                continue 

            prev_frame_arr = prev_frame_arr[0]
            this_frame_arr = this_frame_arr[0]
            frame = int(this_frame_arr[0])

            x0, y0 = t_window_arr[0, 1:3]
            xi, yi = prev_frame_arr[1:3]
            xf, yf = this_frame_arr[1:3]

            segment_length = np.linalg.norm([xf - xi, yf - yi])
            euc_dist = np.linalg.norm([xf - x0, yf - y0])
            speed = segment_length / sampling_interval

            dist_list = [np.linalg.norm(t_window_arr[i + 1, 1:3] - t_window_arr[i, 1:3])
                         for i in range(len(t_window_arr) - 1)]
            cumulative_length = np.sum(dist_list)
            max_dist = np.max(dist_list) if dist_list else 0
            msd = np.sum(np.square(dist_list)) / len(dist_list) if dist_list else 0
            meandering_ind = euc_dist / cumulative_length if cumulative_length != 0 else 0
            outreach_ratio = max_dist / cumulative_length if cumulative_length != 0 else 0
            arrest_coefficient = sum(d < arrest_threshold for d in dist_list) / len(dist_list) if dist_list else 0

            if dist_list and len(dist_list) > 1:
                speed_list = np.array(dist_list) / sampling_interval               
                accel_list = np.diff(speed_list) / sampling_interval               
                avg_acceleration = float(np.mean(np.abs(accel_list)))              
            else:
                avg_acceleration = 0.0

            df_window = pd.DataFrame(t_window_arr, columns=['frame', 'x_position', 'y_position', 'label'])
            neighbor_similarity = calculate_neighbor_similarity(df_window, df_in)
            final_x, final_y = t_window_arr[-1, 1:3]

            loop_features = calculate_loop_features(t_window_arr[:, 1:3])
            
            mig_calcs = [
                str(this_frame_arr[3]), frame, euc_dist, segment_length, cumulative_length, speed,
                meandering_ind, outreach_ratio, msd, max_dist, arrest_coefficient,
                turn_count, neighbor_similarity, final_x, final_y,
                loop_features['loop_score'], loop_features['angle_variation'],
                loop_features['curvature_count'], loop_features['polygon_area'],
                loop_features['turning_index'],
                avg_acceleration 
            ]
            cell_calcs.append(mig_calcs)
    return cell_calcs


def analyze_migration(df_in, t_window=5, sampling_interval=1, arrest_threshold=5, radius=100, min_valid_points=3):
    df_out = df_in.loc[:, ~df_in.columns.duplicated()].copy()
    df_out['label_clean'] = df_out['label'].astype(str).str.strip().str.rstrip('.0')
    df_out['frame'] = df_out['frame'].astype(int)
    df_out['x_position'] = df_out['x_position'].astype(float)
    df_out['y_position'] = df_out['y_position'].astype(float)

    calcs_list = []

    for cell_label in tqdm(df_out['label'].unique()):
        cell_subdf = df_out[df_out['label'] == cell_label]
        tarray = cell_subdf[['frame', 'x_position', 'y_position', 'label']].to_numpy(dtype=float)

        mig_calcs = extract_migration_properties(
            tarray, t_window, sampling_interval,
            arrest_threshold, df_out, min_valid_points
        )
        if mig_calcs:
            calcs_list.extend(mig_calcs)

    if not calcs_list:
        print("No migration features could be extracted.")
        return df_out
    
    mig_df = pd.DataFrame(calcs_list, columns=[
        'label', 'frame', 'euclidean_dist', 'segment_length', 'cumulative_length', 'speed',
        'meandering_index', 'outreach_ratio', 'MSD', 'max_dist', 'arrest_coefficient',
        'turn_count', 'neighbor_similarity', 'final_x', 'final_y',
        'loop_score', 'angle_variation', 'curvature_count', 'polygon_area', 'turning_index',
        'avg_acceleration'
    ])
    return mig_df



In [ ]:
import os
import glob
import pandas as pd
from tqdm import tqdm


def trim_to_first_window(df, t_window=60, sampling_interval=1):
    trimmed_list = []
    for label in df['label'].unique():
        cell_df = df[df['label'] == label].sort_values(by='frame')
        if len(cell_df) >= t_window:
            start_frame = cell_df['frame'].values[0]
            end_frame = start_frame + (t_window - 1) * sampling_interval
            trimmed_df = cell_df[
                (cell_df['frame'] >= start_frame) &
                (cell_df['frame'] <= end_frame)
            ].copy()
            trimmed_list.append(trimmed_df)
    return pd.concat(trimmed_list, ignore_index=True) if trimmed_list else pd.DataFrame()

# --------------------------------------------------------
# Cut the frames from the whole trajectories
# --------------------------------------------------------
def make_segment_df(df_60, seg_idx, seg_len=20, sampling_interval=1):

    # ex) seg_idx=1: 0~19, seg_idx=2: 20~39, seg_idx=3: 40~59
    seg_start_offset = (seg_idx - 1) * seg_len
    seg_end_offset = seg_start_offset + (seg_len - 1)

    seg_list = []
    for label, cell_df in df_60.groupby('label'):
        cell_df = cell_df.sort_values('frame')
        start_frame = cell_df['frame'].iloc[0]

        frame_start = start_frame + seg_start_offset * sampling_interval
        frame_end   = start_frame + seg_end_offset * sampling_interval

        seg_df = cell_df[
            (cell_df['frame'] >= frame_start) &
            (cell_df['frame'] <= frame_end)
        ].copy()

        # If there is no data in segment, skip
        if seg_df.empty:
            continue

        seg_list.append(seg_df)

    if not seg_list:
        return pd.DataFrame()

    seg_df_all = pd.concat(seg_list, ignore_index=True)
    return seg_df_all


# --------------------------------------------------------
# Main loop
# --------------------------------------------------------

# Root folder for whold dataset
root_folder = r"C:\Users\oxfil\Data_output_1127\Data_output_1127" 

for condition_folder in sorted(os.listdir(root_folder)):
    full_condition_path = os.path.join(root_folder, condition_folder)
    if not os.path.isdir(full_condition_path):
        continue

    print(f"\n Current condition: {condition_folder}")

    # Gathering datas
    all_combined_seg1 = []
    all_combined_seg2 = []
    all_combined_seg3 = []

    csv_list = glob.glob(os.path.join(full_condition_path, "*.csv"))
    if not csv_list:
        print(f"{condition_folder} - NO CSV File")
        continue

    for file_path in csv_list:
        filename = os.path.basename(file_path).replace(".csv", "")
        suffix = f"{condition_folder}_{filename}"

        print(f"\n Processing: {suffix}")
        try:
            df = pd.read_csv(file_path)
        except Exception as e:
            print(f"{filename} - fail to get: {e}")
            continue

        # Check required column
        required_cols = ['trajectory_label', 'frame_id', 'centroid_x', 'centroid_y', 'Color']
        if not set(required_cols).issubset(df.columns):
            print(f"{filename} - fail to get: {required_cols}")
            continue

        df['label'] = df['trajectory_label'].astype(str)
        df['frame'] = df['frame_id'].astype(int)
        df['x_position'] = df['centroid_x']
        df['y_position'] = df['centroid_y']

        df_60 = trim_to_first_window(df, t_window=90)
        if df_60.empty:
            print(f"{filename} - No valid data")
            continue

        used_labels_all = df_60['label'].astype(str).unique()

        color_mode_per_label = (
            df[
                df['frame'].isin(df_60['frame'].unique()) &
                df['label'].isin(used_labels_all)
            ]
            .groupby('label')['Color']
            .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else 'Unknown')
            .reset_index()
        )

        # label unite
        color_mode_per_label['label'] = color_mode_per_label['label'].astype(str)

        # === Repeat for each seg ==
        seg_info = {
            1: ("1-30", all_combined_seg1),
            2: ("30-60", all_combined_seg2),
            3: ("60-90", all_combined_seg3),
        }

        for seg_idx, (seg_name, seg_list_collect) in seg_info.items():
            print(f"Segment {seg_idx} ({seg_name}) analyzing..")

            # Segment data set
            seg_df = make_segment_df(df_60, seg_idx=seg_idx, seg_len=30)
            if seg_df.empty:
                print(f"{filename} - Segment {seg_idx} ({seg_name}) no valid trajectories")
                continue

            # Columns for analysis
            seg_df_trimmed = seg_df[['frame', 'x_position', 'y_position', 'label']].copy()

            try:
                df_mig_result = analyze_migration(seg_df_trimmed, t_window=20)
            except Exception as e:
                print(f"{filename} - Segment {seg_idx} analyze_migration 에러: {e}")
                continue

            if df_mig_result is None or 'speed' not in df_mig_result.columns:
                print(f"{filename} - Segment {seg_idx} migration")
                continue

            # mobility summary
            mobility_summary = (
                df_mig_result.groupby('label')
                .agg({
                    'speed': 'mean',
                    'euclidean_dist': 'last',
                    'segment_length': 'mean',
                    'cumulative_length': 'last',
                    'meandering_index': 'mean',
                    'outreach_ratio': 'mean',
                    'MSD': 'mean',
                    'max_dist': 'max',
                    'arrest_coefficient': 'mean',
                    'turn_count': 'last',
                    'neighbor_similarity': 'mean',
                    'loop_score': 'mean',
                    'angle_variation': 'mean',
                    'curvature_count': 'mean',
                    'polygon_area': 'mean',
                    'turning_index': 'mean',
                    'avg_acceleration': 'mean'
                })
                .reset_index()
            )

            if mobility_summary.empty:
                print(f"{filename} - Segment {seg_idx} mobility_summary empty")
                continue

            # label
            mobility_summary['label'] = mobility_summary['label'].astype(float).astype(int).astype(str)

            used_labels_seg = mobility_summary['label'].unique()
            color_summary = color_mode_per_label[color_mode_per_label['label'].isin(used_labels_seg)].copy()

            if color_summary.empty:
                print(f"{filename} - Segment {seg_idx} no Color label")
                continue

            combined_df = pd.merge(color_summary, mobility_summary, on='label', how='inner')
            if combined_df.empty:
                print(f"{filename} - Segment {seg_idx} no combined data")
                continue

            combined_df['label'] = combined_df['label'].apply(lambda x: f"{x}_{suffix}")
            combined_df['source_file'] = filename
            combined_df['Condition'] = condition_folder
            combined_df['Segment'] = seg_name 

            seg_list_collect.append(combined_df)
            print(f"{filename} - Segment {seg_idx} ({seg_name}) 셀 수: {len(combined_df)}")

    if all_combined_seg1:
        final_seg1 = pd.concat(all_combined_seg1, ignore_index=True)
        out1 = f"all_samples_color_vs_mobility_{condition_folder}_seg1_1-30.csv"
        final_seg1.to_csv(out1, index=False)
        print(f"Save: {out1} (total_num: {len(final_seg1)})")
    else:
        print(f"{condition_folder} - No valid data for 1-30")

    if all_combined_seg2:
        final_seg2 = pd.concat(all_combined_seg2, ignore_index=True)
        out2 = f"all_samples_color_vs_mobility_{condition_folder}_seg2_30-60.csv"
        final_seg2.to_csv(out2, index=False)
        print(f"Save: {out2} (total_num: {len(final_seg2)})")
    else:
        print(f"{condition_folder} -  No valid data for 30-60")

    if all_combined_seg3:
        final_seg3 = pd.concat(all_combined_seg3, ignore_index=True)
        out3 = f"all_samples_color_vs_mobility_{condition_folder}_seg3_60-90.csv"
        final_seg3.to_csv(out3, index=False)
        print(f"Save: {out3} (total_num: {len(final_seg3)})")
    else:
        print(f"{condition_folder} - No valid data for 60-90")
